In [1]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg, to_date, date_format

spark = SparkSession.builder.appName("ExpenseMonitoringWeek3").getOrCreate()
print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [2]:
sample_data = """transaction_id,user_id,date,category,amount,description
TXN101,1,2026-01-05,Groceries,120.50,Weekly shopping
TXN102,1,2026-01-12,Utilities,85.00,Electricity bill
TXN103,1,2026-01-20,Groceries,95.00,Supermarket
TXN104,1,2026-01-28,Electronics,2500.00,New Laptop
TXN105,2,2026-01-08,Dining,45.00,Restaurant dinner
TXN106,2,2026-01-15,Rent,1200.00,January house rent
TXN107,2,2026-01-22,Utilities,90.00,Internet bill
TXN108,3,2026-01-10,Groceries,75.00,Groceries
TXN109,3,2026-01-18,Healthcare,60.00,Doctor checkup
TXN110,3,2026-01-25,Jewelry,3200.00,Gold purchase
TXN111,1,2026-02-04,Groceries,130.00,Monthly grocery
TXN112,1,2026-02-14,Entertainment,50.00,Movie night
TXN113,2,2026-02-10,Rent,1200.00,February rent
TXN114,2,2026-02-20,Travel,2800.00,Flight tickets
"""

with open("large_expenses.csv", "w") as f:
    f.write(sample_data.strip())

print("large_expenses.csv created successfully!")

large_expenses.csv created successfully!


In [3]:
# 1. Load the CSV file
df = spark.read.csv("large_expenses.csv", header=True, inferSchema=True)

# 2. Convert date string to date type and extract year-month
df = df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
df = df.withColumn("month", date_format(col("date"), "yyyy-MM"))

print("Schema of loaded data:")
df.printSchema()

print("Sample Transactions:")
df.show(5)

Schema of loaded data:
root
 |-- transaction_id: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- description: string (nullable = true)
 |-- month: string (nullable = true)

Sample Transactions:
+--------------+-------+----------+-----------+------+-----------------+-------+
|transaction_id|user_id|      date|   category|amount|      description|  month|
+--------------+-------+----------+-----------+------+-----------------+-------+
|        TXN101|      1|2026-01-05|  Groceries| 120.5|  Weekly shopping|2026-01|
|        TXN102|      1|2026-01-12|  Utilities|  85.0| Electricity bill|2026-01|
|        TXN103|      1|2026-01-20|  Groceries|  95.0|      Supermarket|2026-01|
|        TXN104|      1|2026-01-28|Electronics|2500.0|       New Laptop|2026-01|
|        TXN105|      2|2026-01-08|     Dining|  45.0|Restaurant dinner|2026-01|
+--------------+-------+

In [4]:
# Calculate total monthly spend and transaction count per user
monthly_spend = df.groupBy("user_id", "month").agg(
    sum("amount").alias("total_monthly_spend"),
    count("transaction_id").alias("transaction_count"),
    avg("amount").alias("average_spend")
).orderBy("user_id", "month")

print("--- Monthly Spend per User ---")
monthly_spend.show()

--- Monthly Spend per User ---
+-------+-------+-------------------+-----------------+------------------+
|user_id|  month|total_monthly_spend|transaction_count|     average_spend|
+-------+-------+-------------------+-----------------+------------------+
|      1|2026-01|             2800.5|                4|           700.125|
|      1|2026-02|              180.0|                2|              90.0|
|      2|2026-01|             1335.0|                3|             445.0|
|      2|2026-02|             4000.0|                2|            2000.0|
|      3|2026-01|             3335.0|                3|1111.6666666666667|
+-------+-------+-------------------+-----------------+------------------+



In [5]:
# Simple threshold logic: any transaction >= 1500 is flagged as an unusual spike
threshold = 1500.0

unusual_expenses = df.filter(col("amount") >= threshold).select(
    "transaction_id",
    "user_id",
    "date",
    "category",
    "amount",
    "description"
).orderBy(col("amount").desc())

print("--- Potential Unusual Spending / Large One-Time Expenses ---")
unusual_expenses.show()

--- Potential Unusual Spending / Large One-Time Expenses ---
+--------------+-------+----------+-----------+------+--------------+
|transaction_id|user_id|      date|   category|amount|   description|
+--------------+-------+----------+-----------+------+--------------+
|        TXN110|      3|2026-01-25|    Jewelry|3200.0| Gold purchase|
|        TXN114|      2|2026-02-20|     Travel|2800.0|Flight tickets|
|        TXN104|      1|2026-01-28|Electronics|2500.0|    New Laptop|
+--------------+-------+----------+-----------+------+--------------+



In [6]:
# Convert to Pandas and save as a single CSV file for submission
output_df = unusual_expenses.toPandas()
output_df.to_csv("unusual_spending_output.csv", index=False)

print("Saved unusual_spending_output.csv successfully!")
print(output_df)

Saved unusual_spending_output.csv successfully!
  transaction_id  user_id        date     category  amount     description
0         TXN110        3  2026-01-25      Jewelry  3200.0   Gold purchase
1         TXN114        2  2026-02-20       Travel  2800.0  Flight tickets
2         TXN104        1  2026-01-28  Electronics  2500.0      New Laptop
